<a href="https://colab.research.google.com/github/Haider-Javed/neurofive-task5/blob/main/neurofive_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Load Titanic dataset built into Seaborn
df = sns.load_dataset('titanic')


feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target_col = 'survived'

# Fill missing values
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

X = df[feature_cols]
y = df[target_col]


X_encoded = pd.get_dummies(X, columns=['sex', 'embarked'], drop_first=True)


X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data ready! Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

Data ready! Training samples: 712 | Testing samples: 179


In [13]:
# Train initial Logistic Regression model
original_model = LogisticRegression(max_iter=1000, random_state=42)
original_model.fit(X_train, y_train)

# Predict on test set
y_pred_original = original_model.predict(X_test)

# Display comprehensive classification metrics
print("=== Baseline Model Classification Report ===")
print(classification_report(y_test, y_pred_original, target_names=['Did Not Survive (0)', 'Survived (1)']))

=== Baseline Model Classification Report ===
                     precision    recall  f1-score   support

Did Not Survive (0)       0.81      0.89      0.85       110
       Survived (1)       0.79      0.67      0.72        69

           accuracy                           0.80       179
          macro avg       0.80      0.78      0.79       179
       weighted avg       0.80      0.80      0.80       179



In [14]:
# Define hyperparameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['liblinear', 'lbfgs'],
    'class_weight': [None, 'balanced']
}

# Initialize GridSearch with 5-fold cross-validation
# We optimize for 'f1' score rather than accuracy
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# Run hyperparameter search
grid_search.fit(X_train, y_train)

print("Best Hyperparameters Found:")
print(grid_search.best_params_)

Best Hyperparameters Found:
{'C': 0.1, 'class_weight': 'balanced', 'solver': 'lbfgs'}


In [15]:
# Predict using the best estimator found by GridSearchCV
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# Helper function to collect metrics
def compute_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

# Calculate metrics for both models
original_metrics = compute_metrics(y_test, y_pred_original)
tuned_metrics = compute_metrics(y_test, y_pred_tuned)

# Create comparison DataFrame
comparison_df = pd.DataFrame([original_metrics, tuned_metrics], index=['Original Model', 'Tuned Model'])
comparison_df = comparison_df.round(4)

print("=== Model Performance Comparison ===")
comparison_df

=== Model Performance Comparison ===


,Accuracy,Precision,Recall,F1-Score
Original Model,0.8045,0.7931,0.6667,0.7244
Tuned Model,0.7933,0.7162,0.7681,0.7413


## ⚠️ The Imbalanced Data Trap: Why Standard Accuracy Fails

When working with real-world datasets, class distributions are rarely split 50/50. In credit card fraud, 99.9% of transactions are legitimate and 0.1% are fraudulent. In medical screening, 95% of patients might be healthy while 5% have a disease.

When you evaluate a model on an imbalanced dataset, relying solely on **Accuracy** creates a major blind spot due to three core factors:

---

### 1. The "Lazy Model" Paradox (Majority Class Dominance)

Machine learning models are mathematical optimization engines designed to maximize overall correctness (or minimize loss).

Consider a dataset with **90 Non-Survivors (Class 0)** and **10 Survivors (Class 1)**:
* A model that literally learns nothing and predicts `0` for **every single passenger** gets 90 out of 100 predictions right.
* This "dumb" baseline yields a **90% Accuracy score**.

$$\text{Accuracy} = \frac{\text{True Positives} + \text{True Negatives}}{\text{Total Samples}} = \frac{0 + 90}{100} = 0.90$$

On paper, 90% accuracy looks impressive, but the model is completely useless because it caught **0% of actual survivors** ($\text{Recall} = 0$).

---

### 2. How Machine Learning Loss Functions Get Distorted

Most classification algorithms (like Logistic Regression or Neural Networks) rely on loss functions such as **Binary Cross-Entropy**:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

In this standard formula, **every sample contributes equally to the total loss**:
* Because majority class samples dominate the training dataset, the gradient updates during training are almost entirely driven by majority samples.
* The model prioritizes adjusting its weights to reduce errors on the majority class, largely ignoring the minority class because mistakes there have a tiny impact on the overall average loss.

---

### 3. High Business Cost of False Negatives

In most real-world scenarios, **a False Negative is much more costly than a False Positive**:

* **Cancer Detection:** Predicting a sick patient is healthy (False Negative) can be fatal. Predicting a healthy patient is sick (False Positive) leads to follow-up testing.
* **Fraud Detection:** Missing a $10,000 fraudulent transaction (False Negative) is a direct financial loss. Flagging a legitimate $10 purchase (False Positive) is a mild inconvenience.

Standard accuracy treats a False Positive and a False Negative as having the exact same penalty, which fails to capture real-world business risks.

---

### 💡 The Solution Checklist for Imbalanced Data

1. **Use Better Metrics:** Evaluate using **Precision**, **Recall**, **F1-Score**, or **PR-AUC** instead of Accuracy.
2. **Adjust Class Weights:** Use parameters like `class_weight='balanced'` in scikit-learn. This scales the loss function so that misclassifying a minority sample carries a much higher penalty.
3. **Resampling Techniques:** Use techniques like **SMOTE** (Synthetic Minority Over-sampling Technique) to generate synthetic minority samples, or random undersampling on the majority class.